# AI Entrepreneur Coach, MVP

Trait based business idea recommender. This notebook builds the lab MVP step by step:

1. Load API keys
2. Parse the profile PDF (LinkedIn export format)
3. Extract structured profile (skills, experience, industry)
4. Big Five (TIPI) input
5. Hardcoded business idea list
6. Career best fit ranking
7. Output report (working style summary, ranked ideas, rationale, 90 day roadmap)

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"

print("OPENAI_API_KEY loaded:", bool(OPENAI_API_KEY))
print("COHERE_API_KEY loaded:", bool(COHERE_API_KEY))

OPENAI_API_KEY loaded: True
COHERE_API_KEY loaded: True


## Step 2: Parse the profile PDF

We extract raw text with `pypdf` and split it into these known sections ourselves, no LLM call needed for this step. Cheaper, and each section is inspectable on its own.

Project 3 still needs to handle other CV formats/structures, this parser is written specifically for the LinkedIn export layout.

In [8]:
from pypdf import PdfReader
import re

PROFILE_PDF_PATH = "input/Profile.pdf"

def fix_line_wrapped_hyphens(text: str) -> str:
    # only join when hyphen attaches to the word directly (no space before it), so a real " - " separator stays untouched
    lines = text.split("\n")
    fixed = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if line.endswith("-") and not line.endswith(" -") and i + 1 < len(lines):
            fixed.append(line + lines[i + 1])
            i += 2
        else:
            fixed.append(line)
            i += 1
    return "\n".join(fixed)

reader = PdfReader(PROFILE_PDF_PATH)
raw_profile_text = "\n".join(page.extract_text() for page in reader.pages)
raw_profile_text = fix_line_wrapped_hyphens(raw_profile_text)

print(raw_profile_text[:500])

   
Contact
004915221433472 (Mobile)
zahra.moghaddasi@gmail.com
www.linkedin.com/in/zahra-moghaddasi (LinkedIn)
Top Skills
Mathematics
Google Cloud Platform (GCP)
C++
Languages
Arabic (Full Professional)
Persian (Native or Bilingual)
English (Professional Working)
German (Elementary)
Certifications
Certified Ethical Hacking
telc Deutsch-Test für den Beruf B2
Zertifikat
Certified Data Vault Practitioner 2.0 -
CDVP2
Honors-Awards
CDVP2
Zahra Moghaddasi
Data Engineer
Berlin Metropolitan Area
Summar


In [9]:
LINKEDIN_SECTIONS = [
    "Contact", "Top Skills", "Languages", "Certifications",
    "Honors-Awards", "Summary", "Experience", "Education",
]

def split_linkedin_sections(text: str) -> dict:
    lines = text.split("\n")
    sections = {}
    current = "header"
    buffer = []
    for line in lines:
        stripped = line.strip()
        if re.match(r"^Page \d+ of \d+$", stripped):
            continue
        if stripped in LINKEDIN_SECTIONS:
            sections[current] = "\n".join(buffer).strip()
            current = stripped
            buffer = []
        else:
            buffer.append(line)
    sections[current] = "\n".join(buffer).strip()
    return sections

def extract_profile_header(sections: dict) -> dict:
    # LinkedIn's export puts name, headline, location right before "Summary" with no header word of its own,
    # so it lands at the tail of whatever sidebar section came last, pull it back out here
    section_names = list(sections.keys())
    if "Summary" not in section_names:
        return {}
    prev_section = section_names[section_names.index("Summary") - 1]
    lines = [l for l in sections[prev_section].split("\n") if l.strip()]
    if len(lines) < 3:
        return {}
    name, headline, location = lines[-3], lines[-2], lines[-1]
    sections[prev_section] = "\n".join(lines[:-3]).strip()
    return {"name": name, "headline": headline, "location": location}

profile_sections = split_linkedin_sections(raw_profile_text)
profile_header = extract_profile_header(profile_sections)

print("profile header:", profile_header)
print()
for section_name, content in profile_sections.items():
    print(f"=== {section_name} ===")
    print(content)
    print()

profile header: {'name': 'Zahra Moghaddasi', 'headline': 'Data Engineer', 'location': 'Berlin Metropolitan Area'}

=== header ===


=== Contact ===
004915221433472 (Mobile)
zahra.moghaddasi@gmail.com
www.linkedin.com/in/zahra-moghaddasi (LinkedIn)

=== Top Skills ===
Mathematics
Google Cloud Platform (GCP)
C++

=== Languages ===
Arabic (Full Professional)
Persian (Native or Bilingual)
English (Professional Working)
German (Elementary)

=== Certifications ===
Certified Ethical Hacking
telc Deutsch-Test für den Beruf B2
Zertifikat
Certified Data Vault Practitioner 2.0 -
CDVP2

=== Honors-Awards ===
CDVP2

=== Summary ===
A PhD Senior Data Warehouse Developer with a passion for solving
business challenges using data Teaching. I am excited to apply
my experience and knowledge of data modeling to solve real world
problems. I have more than seven years of experience in the field
of data engineering and data warehousing where I worked on
several projects. My goal is to deliver optimal solutio

## Step 3: Extract structured profile

Now one LLM call, on the clean sectioned text (not the raw PDF), constrained to a fixed JSON schema so the output is always usable, not free text. `skills` includes both the explicit "Top Skills" list and skills reasonably implied by the summary/experience text.

In [10]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

extraction_input = f"""Headline: {profile_header.get('headline', '')}
Location: {profile_header.get('location', '')}

Top Skills:
{profile_sections.get('Top Skills', '')}

Summary:
{profile_sections.get('Summary', '')}

Experience:
{profile_sections.get('Experience', '')}

Education:
{profile_sections.get('Education', '')}
"""

STRUCTURED_PROFILE_SCHEMA = {
    "type": "object",
    "properties": {
        "skills": {"type": "array", "items": {"type": "string"}},
        "industry": {"type": "string"},
        "years_of_experience": {"type": "number"},
        "experience_summary": {"type": "string"},
        "highest_education": {"type": "string"},
    },
    "required": ["skills", "industry", "years_of_experience", "experience_summary", "highest_education"],
    "additionalProperties": False,
}

extraction_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Extract a structured profile from the given LinkedIn sections. skills should include both explicitly listed skills and skills clearly implied by the experience/summary text. years_of_experience should be your best estimate total professional years.",
        },
        {"role": "user", "content": extraction_input},
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "structured_profile",
            "schema": STRUCTURED_PROFILE_SCHEMA,
            "strict": True,
        }
    },
)

structured_profile = json.loads(extraction_response.output_text)
structured_profile["name"] = profile_header.get("name", "")
structured_profile["location"] = profile_header.get("location", "")

print(json.dumps(structured_profile, indent=2))

{
  "skills": [
    "Data Engineering",
    "Data Warehousing",
    "Data Modeling",
    "Business Problem Solving",
    "Google Cloud Platform (GCP)",
    "Mathematics",
    "C++",
    "Database Management",
    "Image Forensics",
    "Pattern Recognition",
    "Website Development",
    "PHP",
    "ASP.NET",
    "Oracle Development",
    "Software Security"
  ],
  "industry": "Data Engineering and Data Warehousing",
  "years_of_experience": 7,
  "experience_summary": "PhD Senior Data Warehouse Developer with a strong background in data engineering and 7+ years of experience. Worked extensively on data modeling and delivering solutions that meet business objectives. Held various roles ranging from data engineer to head of data management, with a solid foundation in teaching and development.",
  "highest_education": "Doctor of Philosophy (PhD) in Image Forensics",
  "name": "Zahra Moghaddasi",
  "location": "Berlin Metropolitan Area"
}


## Step 4: Big Five via TIPI

The real Ten Item Personality Inventory (Gosling, Rentfrow, Swann, 2003). Each item rated 1 to 7 on "I see myself as...":

- **1 = disagree strongly**, this trait pair does not describe you at all
- **4 = neutral**, neither agree nor disagree
- **7 = agree strongly**, this trait pair describes you very well

Two items per trait, one of the pair reverse scored, then averaged to a 1 to 7 trait score.

No Gradio UI yet, so for now we just set the 10 raw answers directly as a variable to keep testing the pipeline end to end. The UI will replace this cell later with 10 sliders.

In [11]:
TIPI_ITEMS = [
    {"id": 1, "text": "Extraverted, enthusiastic", "trait": "extraversion", "reverse": False},
    {"id": 2, "text": "Critical, quarrelsome", "trait": "agreeableness", "reverse": True},
    {"id": 3, "text": "Dependable, self-disciplined", "trait": "conscientiousness", "reverse": False},
    {"id": 4, "text": "Anxious, easily upset", "trait": "neuroticism", "reverse": False},
    {"id": 5, "text": "Open to new experiences, complex", "trait": "openness", "reverse": False},
    {"id": 6, "text": "Reserved, quiet", "trait": "extraversion", "reverse": True},
    {"id": 7, "text": "Sympathetic, warm", "trait": "agreeableness", "reverse": False},
    {"id": 8, "text": "Disorganized, careless", "trait": "conscientiousness", "reverse": True},
    {"id": 9, "text": "Calm, emotionally stable", "trait": "neuroticism", "reverse": True},
    {"id": 10, "text": "Conventional, uncreative", "trait": "openness", "reverse": True},
]

def score_tipi(answers: dict) -> dict:
    trait_scores = {}
    for item in TIPI_ITEMS:
        raw = answers[item["id"]]
        score = 8 - raw if item["reverse"] else raw
        trait_scores.setdefault(item["trait"], []).append(score)
    return {trait: sum(scores) / len(scores) for trait, scores in trait_scores.items()}

# placeholder answers, will come from Gradio sliders once the UI exists
tipi_answers = {1: 6, 2: 2, 3: 6, 4: 3, 5: 6, 6: 2, 7: 5, 8: 2, 9: 5, 10: 2}

big_five_scores = score_tipi(tipi_answers)
print(json.dumps(big_five_scores, indent=2))

{
  "extraversion": 6.0,
  "agreeableness": 5.5,
  "conscientiousness": 6.0,
  "neuroticism": 3.0,
  "openness": 6.0
}


In [ ]:
# placeholder, will come from Gradio inputs once the UI exists
budget_eur = 500
time_available_hours_per_week = 10

print("budget_eur:", budget_eur)
print("time_available_hours_per_week:", time_available_hours_per_week)

## Step 5: Business idea list

8 hand curated entries, loaded from `input/business_ideas.json`, standing in for the Pinecone knowledge base until Project 3. Each entry has enough structure (budget range, time range, skills, ideal trait ranges on the same 1 to 7 TIPI scale) for the fit calculation in the next step to compute against, not just prose for the LLM to interpret loosely.

In [ ]:
BUSINESS_IDEAS_PATH = "input/business_ideas.json"

with open(BUSINESS_IDEAS_PATH) as f:
    BUSINESS_IDEAS = json.load(f)

print(f"{len(BUSINESS_IDEAS)} business ideas loaded from {BUSINESS_IDEAS_PATH}")
for idea in BUSINESS_IDEAS:
    print(f"- {idea['id']}: {idea['name']}")

## Step 6: Career best fit ranking

Fully computed, not asked from the LLM. Four sub-scores per idea:

- **budget fit / time fit:** having more than the idea needs is fine (1.0), having less is a real shortfall and reduces the score proportionally
- **trait fit:** average, across all 5 Big Five traits, of how close the user's score is to the idea's ideal range (1.0 inside the range, linear falloff outside it)
- **skill fit:** fraction of the idea's needed skills that match something in the user's extracted skills (simple case-insensitive substring match, deterministic, no LLM)

Combined with fixed weights into one career best fit percentage, so it stays traceable back to a formula (see `PROJECT_PLAN.md` success metrics).

In [ ]:
def lower_bound_fit(value: float, low: float, high: float) -> float:
    if low == 0 or value >= low:
        return 1.0
    return max(0.0, value / low)

def range_fit(value: float, low: float, high: float, max_scale: float = 6) -> float:
    if low <= value <= high:
        return 1.0
    distance = low - value if value < low else value - high
    return max(0.0, 1 - distance / max_scale)

def skill_fit(user_skills: list, idea_skills: list) -> float:
    if not idea_skills:
        return 1.0
    user_skills_lower = [s.lower() for s in user_skills]
    matches = sum(
        1 for idea_skill in idea_skills
        if any(idea_skill.lower() in us or us in idea_skill.lower() for us in user_skills_lower)
    )
    return matches / len(idea_skills)

FIT_WEIGHTS = {"budget": 0.2, "time": 0.2, "trait": 0.35, "skill": 0.25}

def compute_career_best_fit(idea: dict) -> dict:
    b_fit = lower_bound_fit(budget_eur, *idea["budget_range_eur"])
    t_fit = lower_bound_fit(time_available_hours_per_week, *idea["time_range_hours_per_week"])
    trait_fits = [range_fit(big_five_scores[trait], *bounds) for trait, bounds in idea["ideal_traits"].items()]
    tr_fit = sum(trait_fits) / len(trait_fits)
    s_fit = skill_fit(structured_profile["skills"], idea["skills_needed"])
    overall = (
        FIT_WEIGHTS["budget"] * b_fit
        + FIT_WEIGHTS["time"] * t_fit
        + FIT_WEIGHTS["trait"] * tr_fit
        + FIT_WEIGHTS["skill"] * s_fit
    )
    return {
        "id": idea["id"],
        "name": idea["name"],
        "description": idea["description"],
        "budget_fit": round(b_fit, 2),
        "time_fit": round(t_fit, 2),
        "trait_fit": round(tr_fit, 2),
        "skill_fit": round(s_fit, 2),
        "career_best_fit_percentage": round(overall * 100, 1),
    }

ranked_ideas = sorted(
    (compute_career_best_fit(idea) for idea in BUSINESS_IDEAS),
    key=lambda r: r["career_best_fit_percentage"],
    reverse=True,
)

top_ideas = ranked_ideas[:5]

for r in top_ideas:
    print(f"{r['career_best_fit_percentage']}% - {r['name']} (budget={r['budget_fit']}, time={r['time_fit']}, trait={r['trait_fit']}, skill={r['skill_fit']})")

## Step 7: Output report

One LLM call writes only the narrative parts (working style summary, per idea rationale, 90 day roadmap for the top match). The fit percentages themselves are NOT touched here, they were already computed in Step 6. Each rationale is grounded by feeding in the specific `matched_skills` and `in_range_traits` for that idea, so the model has to reference real data instead of inventing a connection.

In [16]:
def matched_skills(user_skills: list, idea_skills: list) -> list:
    user_skills_lower = [s.lower() for s in user_skills]
    return [
        idea_skill for idea_skill in idea_skills
        if any(idea_skill.lower() in us or us in idea_skill.lower() for us in user_skills_lower)
    ]

def in_range_traits(idea: dict) -> list:
    return [trait for trait, bounds in idea["ideal_traits"].items() if bounds[0] <= big_five_scores[trait] <= bounds[1]]

ideas_by_id = {idea["id"]: idea for idea in BUSINESS_IDEAS}

grounded_top_ideas = []
for r in top_ideas:
    idea = ideas_by_id[r["id"]]
    grounded_top_ideas.append({
        **r,
        "matched_skills": matched_skills(structured_profile["skills"], idea["skills_needed"]),
        "in_range_traits": in_range_traits(idea),
    })

print(json.dumps(grounded_top_ideas, indent=2))

[
  {
    "id": "idea_001",
    "name": "Freelance Data Engineering / Data Warehousing Consulting",
    "description": "Help small and mid-size companies design pipelines, data models, and warehouses on a contract basis.",
    "budget_fit": 1.0,
    "time_fit": 1.0,
    "trait_fit": 0.97,
    "skill_fit": 0.4,
    "career_best_fit_percentage": 83.8,
    "matched_skills": [
      "data engineering",
      "data modeling"
    ],
    "in_range_traits": [
      "openness",
      "conscientiousness",
      "agreeableness",
      "neuroticism"
    ]
  },
  {
    "id": "idea_008",
    "name": "No-Code Automation / Workflow Consulting for Small Businesses",
    "description": "Help small businesses automate manual processes using no-code tools.",
    "budget_fit": 1.0,
    "time_fit": 1.0,
    "trait_fit": 1.0,
    "skill_fit": 0.25,
    "career_best_fit_percentage": 81.2,
    "matched_skills": [
      "problem solving"
    ],
    "in_range_traits": [
      "openness",
      "conscientiousness

In [ ]:
report_input = f"""User profile:
Name: {structured_profile['name']}
Industry: {structured_profile['industry']}
Years of experience: {structured_profile['years_of_experience']}
Skills: {', '.join(structured_profile['skills'])}
Experience summary: {structured_profile['experience_summary']}

Big Five scores (1-7 scale): {json.dumps(big_five_scores)}
Budget: €{budget_eur}
Time available: {time_available_hours_per_week} hours/week

Top ranked business ideas (already ranked and scored, do not change the ranking or invent a different fit number):
{json.dumps(grounded_top_ideas, indent=2)}
"""

OUTPUT_REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "working_style_summary": {"type": "string"},
        "idea_rationales": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"id": {"type": "string"}, "rationale": {"type": "string"}},
                "required": ["id", "rationale"],
                "additionalProperties": False,
            },
        },
        "roadmap_90_day": {
            "type": "object",
            "properties": {
                "days_1_30": {"type": "string"},
                "days_31_60": {"type": "string"},
                "days_61_90": {"type": "string"},
            },
            "required": ["days_1_30", "days_31_60", "days_61_90"],
            "additionalProperties": False,
        },
    },
    "required": ["working_style_summary", "idea_rationales", "roadmap_90_day"],
    "additionalProperties": False,
}

report_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": (
                "You write the narrative parts of a business idea recommendation report. "
                "Address the person directly as \"you\"/\"your\" throughout, like a coach talking to them, not by their name and not in the third person "
                "(write \"you should focus on...\", not \"Zahra should focus on...\"). "
                "The fit percentages are already computed, do not change or restate them as your own judgment. "
                "working_style_summary: 2 to 4 sentences on the person's working style, based on their Big Five scores and experience. "
                "idea_rationales: one entry for EVERY idea in the list, all of them, each a 1 to 2 sentence rationale that references at least two concrete "
                "things from the data (matched_skills and/or in_range_traits, name the trait), do not invent skills or traits not present in the data. "
                "roadmap_90_day: a 90 day roadmap ONLY for the first (top ranked) idea in the list, one entry per phase (days_1_30, days_31_60, days_61_90). "
                "Each phase should be a full paragraph (5 to 8 sentences), not a one-liner: include concrete first actions, and name specific "
                "real world places or platforms the person could actually use (for example Upwork, Fiverr, LinkedIn, local meetup or coworking groups, "
                "relevant subreddits or Slack/Discord communities, industry conferences), matched to the idea's category."
            ),
        },
        {"role": "user", "content": report_input},
    ],
    text={"format": {"type": "json_schema", "name": "output_report", "schema": OUTPUT_REPORT_SCHEMA, "strict": True}},
)

report_narrative = json.loads(report_response.output_text)
print(json.dumps(report_narrative, indent=2))

In [ ]:
rationales_by_id = {r["id"]: r["rationale"] for r in report_narrative["idea_rationales"]}

final_report_lines = []
final_report_lines.append(f"# AI Entrepreneur Coach report for {structured_profile['name']}\n")
final_report_lines.append("## Working style summary\n")
final_report_lines.append(report_narrative["working_style_summary"] + "\n")
final_report_lines.append("## Ranked business ideas\n")
for r in grounded_top_ideas:
    final_report_lines.append(f"### {r['name']}, career best fit {r['career_best_fit_percentage']}%")
    final_report_lines.append(r["description"])
    final_report_lines.append(rationales_by_id.get(r["id"], "") + "\n")
final_report_lines.append(f"## 90 day roadmap: {grounded_top_ideas[0]['name']}\n")
final_report_lines.append("### Days 1-30")
final_report_lines.append(report_narrative["roadmap_90_day"]["days_1_30"] + "\n")
final_report_lines.append("### Days 31-60")
final_report_lines.append(report_narrative["roadmap_90_day"]["days_31_60"] + "\n")
final_report_lines.append("### Days 61-90")
final_report_lines.append(report_narrative["roadmap_90_day"]["days_61_90"])

final_report = "\n".join(final_report_lines)
print(final_report)

## Step 8: Export the report as a PDF

`fpdf2`, pure Python, no native/system dependencies, safe to install quickly for the lab. Sanitizes a few unicode punctuation marks (curly quotes, em/en dash, ellipsis) to their ASCII equivalents since the built-in Helvetica font is latin-1 only.

In [ ]:
import os
from fpdf import FPDF

def sanitize_for_pdf(text: str) -> str:
    replacements = {"‘": "'", "’": "'", "“": '"', "”": '"', "–": "-", "—": "-", "…": "..."}
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text.encode("latin-1", "replace").decode("latin-1")

pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

def pdf_heading(text: str, size: int = 13) -> None:
    pdf.set_font("Helvetica", "B", size)
    pdf.multi_cell(0, 8, sanitize_for_pdf(text))
    pdf.ln(1)

def pdf_body(text: str, size: int = 11) -> None:
    pdf.set_font("Helvetica", "", size)
    pdf.multi_cell(0, 6, sanitize_for_pdf(text))
    pdf.ln(1)

pdf_heading(f"AI Entrepreneur Coach report for {structured_profile['name']}", size=16)
pdf.ln(2)

pdf_heading("Working style summary")
pdf_body(report_narrative["working_style_summary"])
pdf.ln(2)

pdf_heading("Ranked business ideas")
for r in grounded_top_ideas:
    pdf_heading(f"{r['name']} - career best fit {r['career_best_fit_percentage']}%", size=11)
    pdf_body(r["description"])
    pdf_body(rationales_by_id.get(r["id"], ""))
    pdf.ln(2)

pdf_heading(f"90 day roadmap: {grounded_top_ideas[0]['name']}")
pdf_heading("Days 1-30", size=11)
pdf_body(report_narrative["roadmap_90_day"]["days_1_30"])
pdf_heading("Days 31-60", size=11)
pdf_body(report_narrative["roadmap_90_day"]["days_31_60"])
pdf_heading("Days 61-90", size=11)
pdf_body(report_narrative["roadmap_90_day"]["days_61_90"])

os.makedirs("output", exist_ok=True)
PDF_OUTPUT_PATH = "output/entrepreneur_coach_report.pdf"
pdf.output(PDF_OUTPUT_PATH)

print("Saved to", PDF_OUTPUT_PATH)